# Phase 1 (Eikonal) — CovariateConditionedRanders on Sim2Real-Fire (Flat Grid)

**Scientific question:** Can a Lagrangian covariate-conditioned Randers metric, trained via differentiable geodesic Eikonal solvers, predict wildfire arrival times with Pearson r ≥ 0.70, and does the Finsler wind drift term provide a statistically significant improvement over the isotropic Riemannian baseline?

**Reference:** Gahtan, Shpund & Bronstein (2026). *Wildfire Simulation with Differentiable Randers-Finsler Eikonal Solvers.* arXiv:2603.00035.

---

## Architecture at a glance

```
CovariateConditionedRanders(x, v) = √[gᵢⱼ(x)vⁱvʲ] + bᵢ(x)vⁱ
```
where `g` (PSD matrix field) and `b` (drift/wind vector field) are outputs of a CNN
that takes terrain rasters (elevation, slope, aspect, canopy, fuel) and a weather vector.
The geodesic arc length from ignition to each burned pixel is regressed against the
observed pixel arrival time via the **Eikonal** boundary-value solver with implicit differentiation.

---

## Training loss — curriculum blend

Training uses a **curriculum loss** that ramps from scale-invariant Pearson-r to Relative MSE:

```
L(α) = (1 − α) · (1 − Pearson-r)  +  α · RelMSE
```

* **Warmup** (`α = 0`): pure Pearson-r teaches correct ordering without scale pressure.
* **Ramp** (`α → 1`): Relative MSE forces the model to learn absolute timing — required for IoU.
* α is computed each epoch via `curriculum_alpha(epoch, warmup_epochs, ramp_epochs)`.

---

## Evaluation metrics (Gahtan-comparable)

| Metric | This notebook | Gahtan (2026) target |
|--------|--------------|----------------------|
| **Pearson r** | linear arrival-time correlation | ≈ 0.824 (within-scene) |
| **Spearman r** | rank correlation (scale-invariant) | ≈ 0.695 (cross-scene) |
| **IoU@50** | calibrated spatial overlap (threshold 0.5) | ≈ 0.609 (cross-scene) |

IoU uses a post-hoc scalar `s = mean(GT) / mean(pred)` to map arc-lengths to the GT `[0,1]` time scale before thresholding — matching Gahtan's definition exactly.

---

## Runtime guide

| Mode | Data needed | ~Time (T4 GPU) |
|------|------------|----------------|
| `QUICK = True` + `USE_SYNTHETIC = True` | none | ~5 min |
| `QUICK = False` + `USE_SYNTHETIC = True` | none | ~30 min |
| `QUICK = False` + `USE_SYNTHETIC = False` | Sim2Real-Fire dataset | 2–8 h per scene |

In [ ]:
# ============================================================
# CELL 1 — Colab environment setup
# Run this first. Runtime → Change runtime type → GPU (T4 or A100)
# ============================================================

import subprocess, sys, os

def _run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr[-2000:])
    return result.returncode == 0

# --- Install JAX with CUDA support ---
print("Installing JAX (CUDA 12)...")
_run("pip install -q --upgrade 'jax[cuda12]'")

# --- Install HAMTools (pinned to the active development branch) ---
print("Uninstalling existing HAMTools (if any)...")
_run("pip uninstall -y ham")
print("Installing HAMTools (branch: review/agentic-review)...")
_run("pip install -q 'git+https://github.com/hubibala/ham.git@review/agentic-review'")

# --- Clone / refresh experiment scripts ---
# The notebook imports helpers directly from experiment_eikonal_flat.py.
# Keep the clone in sync with the same branch.
CLONE_DIR = "/content/HAM"
if os.path.isdir(CLONE_DIR):
    print("Refreshing repo clone...")
    _run(f"git -C {CLONE_DIR} fetch --quiet origin review/agentic-review")
    _run(f"git -C {CLONE_DIR} checkout --quiet review/agentic-review")
    _run(f"git -C {CLONE_DIR} pull --quiet origin review/agentic-review")
else:
    print("Cloning HAMTools repo...")
    _run(f"git clone --depth=1 --branch review/agentic-review https://github.com/hubibala/ham.git {CLONE_DIR}")

# Option B: if you cloned the repo to Google Drive, uncomment and use instead:
# import sys; sys.path.insert(0, '/content/drive/MyDrive/HAM/src')

# --- Wildfire optional dependencies ---
print("Installing rasterio + Pillow (wildfire data loaders)...")
_run("pip install -q 'Pillow>=9.0' 'rasterio>=1.3'")

print("Done. Restart runtime if JAX was upgraded (Runtime → Restart session).")

Installing JAX (CUDA 12)...
Installing HAMTools...
Installing rasterio + Pillow (wildfire data loaders)...
Done. Restart runtime if JAX was upgraded (Runtime → Restart session).


In [ ]:
# ============================================================
# CELL 2 — Verify GPU is available
# ============================================================
import jax

# configure_device was added in a recent HAMTools commit.
# If the installed version is older, define it inline here.
try:
    from ham.utils import configure_device
except ImportError:
    def configure_device(device: str):
        dev = jax.devices(device)[0]
        jax.config.update("jax_default_device", dev)
        print(f"[HAMTools inline] Default JAX device set to: {dev}")
        return dev

devices = jax.devices()
print(f"Available JAX devices: {devices}")

# Configure device — change to 'cpu' if no GPU is attached
DEVICE = "gpu" if any("cuda" in str(d).lower() for d in devices) else "cpu"
configure_device(DEVICE)
print(f"Using device: {DEVICE}")

Available JAX devices: [CudaDevice(id=0)]
[HAMTools] Default JAX device set to: cuda:0
Using device: gpu


In [ ]:
# ============================================================
# CELL 3 — Experiment configuration
# ============================================================
import os

# ---- Main switches ----
QUICK         = False  # True = smoke test (~5 min); False = full experiment
USE_SYNTHETIC = False  # True = no dataset needed; False = real Sim2Real-Fire data
USE_WIND      = True   # True = Randers (wind drift); False = Riemannian ablation
OUTPUT_DIR    = "/content/results/phaseEikonal1"

# ---- Dataset location (only used when USE_SYNTHETIC = False) ----
#
# Pick ONE of the three options below and uncomment it.
#
# Option A — Google Drive (recommended for persistent storage)
#   Mount Drive first: Runtime → Mount Drive  (or uncomment the next 2 lines)
# from google.colab import drive; drive.mount("/content/drive")
# DATA_ROOT = "/content/drive/MyDrive/sim2real_fire"
#
# Option B — Clone Gahtan et al. data repo directly into Colab
# !git clone --depth=1 https://github.com/BarakGahtan/differentiable-eikonal-wildfire /content/gahtan
# DATA_ROOT = "/content/gahtan/experiments/wildfire/data"
#
# Option C — Local run (not on Colab)
# DATA_ROOT = "data/sim2real_fire"
#
# ---- Active selection ----
ON_COLAB = "google.colab" in __import__("sys").modules or os.path.exists("/content")
if ON_COLAB:
    DATA_ROOT = "/content/drive/MyDrive/sim2real_fire"   # <-- edit this path if needed
else:
    DATA_ROOT = "data/sim2real_fire"

SCENE_IDS = ["0001"]  # list of scene folder names inside DATA_ROOT

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/figs", exist_ok=True)

if not USE_SYNTHETIC and not os.path.isdir(DATA_ROOT):
    print(f"WARNING: DATA_ROOT does not exist: {DATA_ROOT}")
    print("  → Set USE_SYNTHETIC = True to run without the dataset, or")
    print("  → Mount Google Drive and set DATA_ROOT to the correct path.")
else:
    print(f"Config: QUICK={QUICK}, USE_SYNTHETIC={USE_SYNTHETIC}, USE_WIND={USE_WIND}")
    print(f"DATA_ROOT: {DATA_ROOT}")

Config: QUICK=True, USE_SYNTHETIC=True, USE_WIND=True


In [ ]:
# ============================================================
# CELL 4 — Imports
# ============================================================
import os, sys, time
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display

import jax
import jax.numpy as jnp
import equinox as eqx
import optax
from jax import config
config.update("jax_enable_x64", True)

# --- HAMTools core ---
from ham.geometry.manifolds import EuclideanSpace
from ham.solvers.avbd import AVBDSolver
from ham.training.losses import ArrivalTimeLoss

# curriculum_alpha: added in a recent HAMTools commit; define inline if absent.
try:
    from ham.training.losses import curriculum_alpha
except ImportError:
    def curriculum_alpha(epoch: int, warmup_epochs: int, ramp_epochs: int) -> float:
        if epoch < warmup_epochs:
            return 0.0
        return float(min((epoch - warmup_epochs) / max(ramp_epochs, 1), 1.0))
    print("[HAMTools] Using inline fallback for curriculum_alpha")

# --- Locate experiment helpers ---
ON_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
if ON_COLAB:
    # Try the cloned repo path first, then the installed package location
    for _p in ("/content/HAM/examples", "/HAM/examples"):
        if os.path.isdir(_p):
            _examples_path = _p
            break
    else:
        _examples_path = "/content/HAM/examples"
else:
    _examples_path = os.path.abspath(os.path.join(os.path.dirname("file"), ".."))

if _examples_path not in sys.path:
    sys.path.insert(0, _examples_path)

print(f"Looking for experiment scripts in: {_examples_path}")

# Import symbols; fall back to inline definitions for anything added recently.
try:
    from experiment_eikonal_flat import (
        get_config, make_metric, make_solver,
        bind_scenario_to_metric,
        _make_synthetic_scenario,
        _pixels_to_world, _ignition_to_world,
        pearson_r, spearman_r,
        _predict_arrivals_chunked,
        run_synthetic, run_experiment,
    )
    print("Imported experiment_eikonal_flat OK (full)")
except ImportError as e:
    # spearman_r was added in a recent commit; define inline and retry.
    if "spearman_r" in str(e):
        from experiment_eikonal_flat import (
            get_config, make_metric, make_solver,
            bind_scenario_to_metric,
            _make_synthetic_scenario,
            _pixels_to_world, _ignition_to_world,
            pearson_r,
            _predict_arrivals_chunked,
            run_synthetic, run_experiment,
        )

        def spearman_r(a, b):
            """Rank correlation — inline fallback (update the repo clone to get the real version)."""
            a = np.asarray(a, dtype=np.float64)
            b = np.asarray(b, dtype=np.float64)
            if len(a) < 2:
                return 0.0
            rank_a = np.argsort(np.argsort(a)).astype(np.float64)
            rank_b = np.argsort(np.argsort(b)).astype(np.float64)
            return pearson_r(rank_a, rank_b)

        print("Imported experiment_eikonal_flat OK (with inline spearman_r fallback)")
        print("  → To remove the fallback: git pull origin review/agentic-review in the Colab clone.")
    else:
        print(f"Import error from local path: {e}")
        print(f"Contents of {_examples_path}:")
        print([f for f in os.listdir(_examples_path) if f.endswith(".py")])

Looking for experiment scripts in: /HAM/
Import error: No module named 'experiment_wildfire_flat'
Contents of /HAM/:


FileNotFoundError: [Errno 2] No such file or directory: '/HAM/'

In [ ]:
# ============================================================
# CELL 5 — Build configuration
# ============================================================
cfg = get_config(quick=QUICK)

# Adjust batch size for GPU memory
if DEVICE == "gpu":
    cfg["batch_size_fires"] = 32 if not QUICK else 16
    cfg["k_train_obs"] = 200 if not QUICK else 100

print("Configuration:")
for k, v in cfg.items():
    print(f"  {k:35s} = {v}")

In [ ]:
# ============================================================
# CELL 6 — Run experiment
# ============================================================
if USE_SYNTHETIC:
    print("Running synthetic smoke test (no real dataset needed)...")
    result = run_synthetic(cfg, output_dir=OUTPUT_DIR, use_wind=USE_WIND)
    all_results = [result]
else:
    print(f"Running real-data experiment on scenes: {SCENE_IDS}")
    all_results = run_experiment(
        data_root=DATA_ROOT,
        scene_ids=SCENE_IDS,
        output_dir=OUTPUT_DIR,
        cfg=cfg,
        use_wind=USE_WIND,
    )

print("\n=== Final Results ===")
for r in all_results:
    spr  = r.get("test_spearman_r_mean", float("nan"))
    cov  = r.get("eval_coverage", float("nan"))
    iou  = r.get("test_iou50", float("nan"))
    wind = "yes" if r.get("use_wind") else "no"
    print(
        f"  scene={r['scene_id']}  "
        f"Pearson r={r['test_pearson_r_mean']:.4f}±{r['test_pearson_r_std']:.4f}  "
        f"Spearman r={spr:.4f}  "
        f"IoU@50={iou:.4f} (cov={cov:.1%})  "
        f"wind={wind}"
    )
print()
print("Gahtan et al. (2026) baselines:")
print("  Pearson r  ≈ 0.824  (within-scene)  /  0.733 (cross-scene)")
print("  Spearman r ≈ 0.695  (cross-scene)   /  0.696 (sim-to-real)")
print("  IoU@50     ≈ 0.609  (cross-scene)")

Running real-data experiment on scenes: ['0001']

Scene 0001  seed=0  wind=yes
  Events: 2614 total | 1830 train / 392 val / 392 test
  Fitting SceneNormalizer on training fires...
  Loading & preprocessing scenarios (n_workers=8)...
  Loaded 1830 / 392 / 392 train/val/test scenarios
  Batched training: B=32 fires/step, 57 steps/epoch, vmap kernel size=6400 obs-paths
  Epoch   1/100: loss=0.30222  val_r=0.7717  alpha=0.00  time=454.6s
  Epoch   2/100: loss=0.22874  val_r=0.7780  alpha=0.00  time=432.2s
  Epoch   3/100: loss=0.22605  val_r=0.7732  alpha=0.00  time=431.2s
  Epoch   4/100: loss=0.25973  val_r=0.6362  alpha=0.00  time=430.9s
  Epoch   5/100: loss=0.35899  val_r=0.6772  alpha=0.00  time=431.4s
  Epoch   6/100: loss=0.35418  val_r=0.6814  alpha=0.00  time=431.9s
  Epoch   7/100: loss=0.32394  val_r=0.6905  alpha=0.05  time=431.7s
  Epoch   8/100: loss=0.32930  val_r=0.6059  alpha=0.10  time=428.4s
  Epoch   9/100: loss=0.33047  val_r=0.6456  alpha=0.15  time=431.2s
  Epoch  

In [ ]:
# ============================================================
# CELL 7 — Training loss convergence
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for r in all_results[:3]:
    label = f"{r['scene_id']} seed={r.get('seed', '?')}"
    epochs = np.arange(1, len(r["train_loss_history"]) + 1)
    axes[0].plot(epochs, r["train_loss_history"], label=label)
    if r.get("val_r_history"):
        axes[1].plot(np.arange(1, len(r["val_r_history"]) + 1),
                     r["val_r_history"], label=label)

axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Train loss (MSE)")
axes[0].set_title("Loss convergence"); axes[0].legend(fontsize=8)
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Val Pearson r")
axes[1].set_title("Validation correlation"); axes[1].legend(fontsize=8)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/figs/eikonal1_convergence.png", dpi=120)
plt.show()

In [ ]:
# ============================================================
# CELL 8 — Per-scene metric bar charts (Pearson r + Spearman r + IoU@50)
# ============================================================
scene_ids_unique = sorted({r["scene_id"] for r in all_results})

def _scene_stats(key, default=float("nan")):
    means = [np.mean([r.get(key, default) for r in all_results if r["scene_id"] == sid])
             for sid in scene_ids_unique]
    stds  = [np.std([r.get(key, default) for r in all_results if r["scene_id"] == sid])
             for sid in scene_ids_unique]
    return means, stds

r_means,   r_stds   = _scene_stats("test_pearson_r_mean")
spr_means, spr_stds = _scene_stats("test_spearman_r_mean")
iou_means, _        = _scene_stats("test_iou50")
cov_means, _        = _scene_stats("eval_coverage", default=1.0)

x = np.arange(len(scene_ids_unique))
w = 0.35

fig, axes = plt.subplots(1, 2, figsize=(max(10, len(scene_ids_unique) * 2.2), 4.5))

# --- Left: Pearson r and Spearman r side-by-side ---
ax = axes[0]
ax.bar(x - w/2, r_means,   w, yerr=r_stds,   capsize=4, color="#4477AA", alpha=0.85, label="Pearson r")
ax.bar(x + w/2, spr_means, w, yerr=spr_stds, capsize=4, color="#66BBAA", alpha=0.85, label="Spearman r")
ax.axhline(0.824, color="#CC6677", linestyle="--", linewidth=1.2, label="Gahtan Pearson (0.824)")
ax.axhline(0.695, color="#44AA99", linestyle=":",  linewidth=1.2, label="Gahtan Spearman (0.695)")
ax.axhline(0.70,  color="grey",    linestyle="--", linewidth=1.0, alpha=0.5, label="Target r=0.70")
ax.set_xticks(x); ax.set_xticklabels(scene_ids_unique, rotation=30, ha="right")
ax.set_ylabel("Correlation"); ax.set_ylim(0, 1.05)
ax.set_title("Phase Eikonal1 — Arrival-time correlation")
ax.legend(fontsize=8)

# --- Right: Calibrated IoU@50 ---
ax = axes[1]
ax.bar(x, iou_means, color="#AA3377", alpha=0.85, label="IoU@50 (calibrated)")
ax.axhline(0.609, color="#882255", linestyle="--", linewidth=1.2, label="Gahtan IoU@50 (0.609)")
for xi, (iou_v, cov_v) in enumerate(zip(iou_means, cov_means)):
    ax.text(xi, iou_v + 0.01, f"cov={cov_v:.0%}", ha="center", va="bottom", fontsize=7)
ax.set_xticks(x); ax.set_xticklabels(scene_ids_unique, rotation=30, ha="right")
ax.set_ylabel("IoU@50"); ax.set_ylim(0, 0.75)
ax.set_title("Phase Eikonal1 — Calibrated IoU@50")
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/figs/eikonal1_metrics_bar.png", dpi=120)
plt.show()

print("\nSummary:")
for sid, rm, rs, sp, iou, cov in zip(scene_ids_unique, r_means, r_stds, spr_means, iou_means, cov_means):
    status = "PASS" if rm >= 0.70 else "FAIL"
    print(f"  [{status}] {sid}: Pearson r={rm:.4f}±{rs:.4f}  Spearman r={sp:.4f}  IoU@50={iou:.4f} (cov={cov:.1%})")

In [ ]:
# ============================================================
# CELL 9 — Arrival-time heatmap (synthetic mode only)
# ============================================================
if USE_SYNTHETIC and all_results and "pred_grid" in all_results[0]:
    r = all_results[0]
    pred_grid = np.array(r["pred_grid"])
    gt_grid   = np.array(r["gt_grid"])
    pred_norm = (pred_grid - pred_grid.min()) / (pred_grid.max() - pred_grid.min() + 1e-8)

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    im0 = axes[0].imshow(gt_grid, cmap="hot_r", vmin=0, vmax=1)
    axes[0].set_title("Ground truth arrival time")
    plt.colorbar(im0, ax=axes[0])

    im1 = axes[1].imshow(pred_norm, cmap="hot_r", vmin=0, vmax=1)
    axes[1].set_title(f"Predicted (r={r['test_pearson_r_mean']:.3f})")
    plt.colorbar(im1, ax=axes[1])

    err = np.abs(pred_norm - gt_grid)
    im2 = axes[2].imshow(err, cmap="Reds")
    axes[2].set_title("Absolute error")
    plt.colorbar(im2, ax=axes[2])

    plt.suptitle("Phase Eikonal1 — Synthetic smoke test heatmap", fontsize=12)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/figs/eikonal1_heatmap.png", dpi=120)
    plt.show()
else:
    print("Heatmap requires USE_SYNTHETIC=True and 'pred_grid' key in results.")
    print("For real-data runs, per-scene figures are saved to OUTPUT_DIR/figs/ automatically.")

In [ ]:
# ============================================================
# CELL 10 — Save checkpoint
# ============================================================
import equinox as eqx

CHECKPOINT_PATH = f"{OUTPUT_DIR}/metric_eikonal1_seed{cfg['seed']}.eqx"

if "metric" in dir() and metric is not None:
    eqx.tree_serialise_leaves(CHECKPOINT_PATH, metric)
    print(f"Checkpoint saved: {CHECKPOINT_PATH}")
    print("Load this in the W3 geometric analysis notebook with:")
    print(f"  metric = eqx.tree_deserialise_leaves('{CHECKPOINT_PATH}', metric_template)")
else:
    print("No 'metric' variable found in scope.")
    print("The run_synthetic / run_experiment functions save checkpoints automatically to OUTPUT_DIR.")

# To copy to Google Drive on Colab:
# import shutil
# shutil.copy(CHECKPOINT_PATH, '/content/drive/MyDrive/ham_checkpoints/')